# Case Study Overview

### Contexto
A **SkiWell Sports** é uma empresa de produção
e retalho especializada em artigos de desporto
de inverno, fundada em 2011 em Andorra.
A empresa **desenvolve, fabrica e comercializa**
uma gama completa de **produtos** relacionados
com **ski**, incluindo equipamento técnico, roupa,
acessórios e material de proteção. Ao longo
da última década e meia, a SkiWell Sports
expandiu a sua presença internacional,
operando atualmente em diversos mercados,
nomeadamente Europa, América do Norte
e Ásia.
Apesar do posicionamento sólido e da procura
consistente durante a época de inverno,
surgiram indícios de que a rentabilidade global
poderá ter diminuído. Perante esta suspeita,
a empresa enfrenta agora um desafio
estratégico: avaliar se existe uma diminuição
da rentabilidade entre as diferentes
geografias e identificar gaps de performance
que possam justificar o eventual encerramento
das operações numa geografia.

---

## Load Data

In [9]:
import pandas as pd
import os

# Adjust path as needed
# For Google Colab: data_dir = '/content/Data/'
_nb_dir   = os.path.dirname(os.path.abspath('__file__'))
data_dir  = os.path.join(_nb_dir, '..', '02. Dados')

def load(fname, region):
    df = pd.read_excel(os.path.join(data_dir, fname))
    df['Region'] = region
    return df

europe        = load('Europe_SalesData.xlsx',       'Europe')
north_america = load('North America_SalesData.xlsx','North America')
asia          = load('Asia_SalesData.xlsx',          'Asia')

# Normalise shop-id column names
europe        = europe.rename(columns={'Shop_ID':  'Store_ID'})

# Combine and restrict to 2020-2025
df_all = pd.concat([europe, north_america, asia], ignore_index=True)
df     = df_all[df_all['Year'].between(2020, 2025)].copy()

# Derived columns
df['Profit']  = df['Sales_Price'] - df['Production_Cost'] - df['Shipping_Cost'].fillna(0)
df['Revenue'] = df['Sales_Price']

print(f"Total de registos (2020-2025): {len(df):,}\n")
print(df.groupby('Region').size().rename('Registos'))


Total de registos (2020-2025): 147,366

Region
Asia             36283
Europe           74126
North America    36957
Name: Registos, dtype: int64


# 1. Análise Global

### Tópicos de Finanças (Global)

**Receita e Lucratividade**: A empresa apresenta uma trajetória de crescimento sólido, com a receita global saltando de cerca de 2,8M€ em 2020 para mais de 6,5M€ em 2025.

**Margens**: A margem de lucro líquido global mantém-se estável e saudável em torno de 43%, indicando uma boa gestão de custos diretos e preços de venda.

**Ponto de Inflexão**: O ano de 2022 foi o de maior crescimento relativo (+56%), sugerindo uma expansão bem-sucedida ou um aumento significativo na procura global.

In [10]:
# ── Receita e Lucro por Ano ────────────────────────────────────────────────────
fin = df.groupby('Year').agg(
    Receita   = ('Revenue', 'sum'),
    Lucro     = ('Profit',  'sum')
).round(0)

fin['Margem (%)']   = (fin['Lucro'] / fin['Receita'] * 100).round(1)
fin['Crescimento (%)'] = fin['Receita'].pct_change().mul(100).round(1)

fin.index = fin.index.astype(str)
fin.loc['Total'] = [
    fin['Receita'].sum(),
    fin['Lucro'].sum(),
    (fin['Lucro'].sum() / fin['Receita'].sum() * 100).round(1),
    float('nan')
]
fin['Receita']   = fin['Receita'].map(lambda x: f'€{x/1e6:.2f}M' if pd.notna(x) else '')
fin['Lucro']     = fin['Lucro'].map(lambda x:   f'€{x/1e6:.2f}M' if pd.notna(x) else '')
fin['Crescimento (%)'] = fin['Crescimento (%)'].map(lambda x: f'+{x:.0f}%' if pd.notna(x) and x >= 0 else (f'{x:.0f}%' if pd.notna(x) else '—'))

display(fin)


,Receita,Lucro,Margem (%),Crescimento (%)
Year,,,,
2020,€2.80M,€1.21M,43.1,—
2021,€3.40M,€1.47M,43.2,+22%
2022,€5.31M,€2.30M,43.3,+56%
2023,€5.75M,€2.48M,43.2,+8%
2024,€6.04M,€2.62M,43.4,+5%
2025,€6.51M,€2.81M,43.1,+8%
Total,€29.80M,€12.88M,43.2,—


### Tópicos de Logística (Global)

**Operadores**: A SkiWell utiliza quatro parceiros principais: SafeTrans, QuickMove, SnowLogistics e FastDelivery.

**Eficiência**: O tempo médio de entrega é consistente entre os operadores, situando-se nos 5 dias. O custo médio de envio por encomenda é de aproximadamente 20€, independentemente da transportadora, o que sugere contratos de logística bem padronizados.

In [11]:
# ── Operadores Logísticos – Volume, Tempo e Custo ─────────────────────────────
has_ship = df[df['Shipping_Company'].notna()].copy()
has_ship['Delivery_Days'] = (has_ship['Due_Date'] - has_ship['Order_Date']).dt.days

log = has_ship.groupby('Shipping_Company').agg(
    Encomendas        = ('Sales_Order_ID',  'count'),
    Tempo_Medio_Dias  = ('Delivery_Days',   'mean'),
    Custo_Medio_EUR   = ('Shipping_Cost',   'mean')
).round(2)

log['Tempo_Medio_Dias'] = log['Tempo_Medio_Dias'].map(lambda x: f'{x:.0f} dias')
log['Custo_Medio_EUR']  = log['Custo_Medio_EUR'].map(lambda x:  f'€{x:.2f}')
log.index.name = 'Operador'

display(log)

# Pct de encomendas sem custo de envio registado
pct_sem_custo = df['Shipping_Cost'].isna().mean() * 100
print(f"\n{pct_sem_custo:.1f}% das encomendas não têm custo de envio registado (possível Pick-up em loja).")


,Encomendas,Tempo_Medio_Dias,Custo_Medio_EUR
Operador,,,
FastDelivery,3517,5 dias,€20.03
QuickMove,8730,5 dias,€20.21
SafeTrans,8905,5 dias,€19.97
SnowLogistics,5294,5 dias,€20.17



82.1% das encomendas não têm custo de envio registado (possível Pick-up em loja).


### Tópicos Operacionais (Global)

**Mix de Produtos**: As categorias com maior volume de vendas são Acessórios (Capacetes) e Vestuário (Casacos). Os capacetes, em particular, representam o maior volume unitário, enquanto os casacos geram o maior valor absoluto de lucro.

**Saturação de Canais**: A Europa opera com o dobro do volume de transações das outras regiões, indicando uma infraestrutura operacional muito mais densa e madura.

In [12]:
# ── Mix de Produtos – Volume e Lucro por Categoria ────────────────────────────
cat = df.groupby('Product_Category').agg(
    Unidades = ('Sales_Order_ID', 'count'),
    Lucro    = ('Profit',         'sum')
).sort_values('Unidades', ascending=False)

cat['Lucro'] = cat['Lucro'].map(lambda x: f'€{x/1e3:.1f}k')
display(cat.head(10))

# ── Transações por Região ──────────────────────────────────────────────────────
print("\n── Transações por Região ────────────────────────────────────")
tx = df.groupby('Region').size().rename('Transações').to_frame()
tx['%'] = (tx['Transações'] / tx['Transações'].sum() * 100).map(lambda x: f'{x:.1f}%')
display(tx)


,Unidades,Lucro
Product_Category,,
Accessories_Helmets,10654,€596.9k
Clothing_Jackets,10604,€912.0k
Accessories_Ski socks,10586,€103.2k
Accessories_Gloves,10569,€230.0k
Clothing_Top base layers,10566,€366.2k
Technical equipment_Skis,10556,€3672.6k
Clothing_Bottom base layers,10528,€747.7k
Accessories_Goggles,10492,€518.9k
Accessories_Neck warmers,10487,€148.0k



── Transações por Região ────────────────────────────────────


,Transações,%
Region,,
Asia,36283,24.6%
Europe,74126,50.3%
North America,36957,25.1%


### Tópicos de Estratégia (Global)

**Dominância de Mercado**: A Europa detém 50,5% do market share total da empresa, sendo o pilar central da estratégia atual.

**Diversificação**: Embora a Europa seja dominante, a dependência excessiva de um único mercado pode ser um risco caso haja invernos menos rigorosos na região.

In [13]:
# ── Market Share por Região (Receita Total 2020-2025) ─────────────────────────
ms = df.groupby('Region').agg(
    Receita = ('Revenue', 'sum'),
    Lucro   = ('Profit',  'sum')
).round(0)

ms['Market Share (%)'] = (ms['Receita'] / ms['Receita'].sum() * 100).round(1)
ms['Margem (%)']       = (ms['Lucro']   / ms['Receita']          * 100).round(1)

ms['Receita'] = ms['Receita'].map(lambda x: f'€{x/1e6:.2f}M')
ms['Lucro']   = ms['Lucro'].map(lambda x:   f'€{x/1e6:.2f}M')
ms['Market Share (%)'] = ms['Market Share (%)'].map(lambda x: f'{x:.1f}%')
ms['Margem (%)']       = ms['Margem (%)'].map(lambda x:       f'{x:.1f}%')

display(ms)


,Receita,Lucro,Market Share (%),Margem (%)
Region,,,,
Asia,€7.30M,€3.18M,24.5%,43.6%
Europe,€15.05M,€6.56M,50.5%,43.6%
North America,€7.45M,€3.13M,25.0%,42.1%


---

# 2. Análise Regional

### Tópicos de Finanças (Regional)

**Europa**: Líder absoluta em rentabilidade (6,5M€ de lucro acumulado no período relevante).

**América do Norte**: Apresenta a margem mais baixa entre as regiões (42.1%), cerca de 1.5 pontos percentuais abaixo da Europa e Ásia, o que sugere custos operacionais ou de distribuição ligeiramente superiores.

**Ásia**: Apesar de ter boas margens (43.6%), o crescimento da receita estagnou entre 2023 e 2024.

In [15]:
# ── Receita por Região e Ano ──────────────────────────────────────────────────
rev_reg = df.pivot_table(values='Revenue', index='Region', columns='Year',
                          aggfunc='sum').round(0)
rev_reg.columns = rev_reg.columns.astype(str)
rev_reg['Total'] = rev_reg.sum(axis=1)

display(rev_reg.map(lambda x: f'€{x/1e3:.0f}k'))

# ── Margem por Região e Ano ───────────────────────────────────────────────────
print("\n── Margem (%) por Região e Ano ──────────────────────────────")
margin_reg = df.groupby(['Region','Year']).apply(
    lambda g: g['Profit'].sum() / g['Revenue'].sum() * 100,
    include_groups=False
).round(1).unstack()
margin_reg.columns = margin_reg.columns.astype(str)

display(margin_reg.map(lambda x: f'{x:.1f}%'))

# ── Estagnação da Ásia 2023-2024 ─────────────────────────────────────────────
print("\n── Ásia: variação de receita YoY ────────────────────────────")
asia_rev = df[df['Region']=='Asia'].groupby('Year')['Revenue'].sum()
asia_yoy = asia_rev.pct_change().mul(100).round(1)
asia_df  = pd.DataFrame({'Receita': asia_rev.map(lambda x: f'€{x/1e3:.0f}k'),
                          'Crescimento (%)': asia_yoy.map(lambda x: f'{x:+.1f}%' if pd.notna(x) else '—')})
display(asia_df)


Year,2020,2021,2022,2023,2024,2025,Total
Region,,,,,,,
Asia,€689k,€765k,€1401k,€1468k,€1443k,€1536k,€7302k
Europe,€1413k,€1753k,€2618k,€2881k,€3076k,€3307k,€15049k
North America,€693k,€885k,€1290k,€1395k,€1517k,€1669k,€7449k



── Margem (%) por Região e Ano ──────────────────────────────


Year,2020,2021,2022,2023,2024,2025
Region,,,,,,
Asia,43.5%,43.6%,43.7%,43.6%,43.5%,43.5%
Europe,43.4%,43.6%,43.7%,43.6%,43.9%,43.5%
North America,42.2%,41.9%,41.9%,42.0%,42.4%,42.0%



── Ásia: variação de receita YoY ────────────────────────────


,Receita,Crescimento (%)
Year,,
2020,€689k,—
2021,€765k,+11.0%
2022,€1401k,+83.2%
2023,€1468k,+4.8%
2024,€1443k,-1.7%
2025,€1536k,+6.4%


### Tópicos de Logística (Regional)

**Regionalização**: Na América do Norte, a SafeTrans é a parceira dominante. Na Ásia, a logística parece estar mais concentrada na SnowLogistics.

**Custos Ocultos**: Observou-se que em certas regiões, uma percentagem considerável de ordens não tem custos de envio registados, o que pode indicar vendas diretas em loja (Pick-up) ou falhas no registo de dados logísticos.

In [16]:
# ── Operadores por Região ─────────────────────────────────────────────────────
ship_reg = (df[df['Shipping_Company'].notna()]
            .groupby(['Region','Shipping_Company'])
            .size()
            .unstack(fill_value=0))
display(ship_reg)

# ── % de Encomendas sem Custo de Envio por Região ─────────────────────────────
print("\n── % Encomendas sem custo de envio (possível Pick-up) ───────")
null_ship = df.groupby('Region').apply(
    lambda g: g['Shipping_Cost'].isna().mean() * 100, include_groups=False
).round(1).rename('% Sem Envio Registado').to_frame()
null_ship['% Sem Envio Registado'] = null_ship['% Sem Envio Registado'].map(lambda x: f'{x:.1f}%')
display(null_ship)


Shipping_Company,FastDelivery,QuickMove,SafeTrans,SnowLogistics
Region,,,,
Asia,0,0,0,5294
Europe,3517,3538,3535,0
North America,0,5192,5370,0



── % Encomendas sem custo de envio (possível Pick-up) ───────


,% Sem Envio Registado
Region,
Asia,85.4%
Europe,85.7%
North America,71.4%


### Tópicos Operacionais (Regional)

**Performance por Loja**: As lojas na Ásia (especialmente as IDs 23, 22 e 24) apresentam os menores volumes de lucro líquido acumulado, contrastando com a alta performance das lojas europeias (IDs 1 a 10).

**Preferências de Produto**: Existe uma consistência global no mix de produtos, sugerindo que a marca SkiWell Sports tem uma aceitação uniforme, sem necessidade de grandes adaptações locais de portfólio.

In [17]:
# ── Lucro por Loja ────────────────────────────────────────────────────────────
shop_profit = df.groupby(['Region','Store_ID'])['Profit'].sum().reset_index()
shop_profit = shop_profit.sort_values('Profit')

print("── 8 Lojas com Menor Lucro Acumulado (2020-2025) ─────────────")
display(shop_profit.head(8).assign(
    Lucro=lambda x: x['Profit'].map(lambda v: f'€{v/1e3:.0f}k')
)[['Region','Store_ID','Lucro']].reset_index(drop=True))

print("\n── 8 Lojas com Maior Lucro Acumulado ─────────────────────────")
display(shop_profit.tail(8).sort_values('Profit', ascending=False).assign(
    Lucro=lambda x: x['Profit'].map(lambda v: f'€{v/1e3:.0f}k')
)[['Region','Store_ID','Lucro']].reset_index(drop=True))

# ── Consistência do Mix de Produtos por Região ────────────────────────────────
print("\n── Top 5 Categorias por Região (por unidades vendidas) ──────")
for region in ['Europe','North America','Asia']:
    top5 = (df[df['Region']==region]
            .groupby('Product_Category')
            .size()
            .sort_values(ascending=False)
            .head(5))
    print(f"\n{region}:")
    print(top5.to_string())


── 8 Lojas com Menor Lucro Acumulado (2020-2025) ─────────────


,Region,Store_ID,Lucro
0,North America,18.0,€469k
1,North America,13.0,€471k
2,Europe,10.0,€472k
3,North America,12.0,€473k
4,Europe,21.0,€475k
5,Europe,8.0,€475k
6,Europe,4.0,€476k
7,North America,15.0,€478k



── 8 Lojas com Maior Lucro Acumulado ─────────────────────────


,Region,Store_ID,Lucro
0,North America,14.0,€753k
1,Europe,2.0,€735k
2,Europe,11.0,€498k
3,Europe,3.0,€497k
4,North America,16.0,€490k
5,Europe,6.0,€490k
6,Europe,20.0,€489k
7,Europe,9.0,€488k



── Top 5 Categorias por Região (por unidades vendidas) ──────

Europe:
Product_Category
Accessories_Goggles         5406
Technical equipment_Skis    5369
Clothing_Jackets            5357
Accessories_Neck warmers    5351
Accessories_Helmets         5313

North America:
Product_Category
Accessories_Helmets         2713
Clothing_Jackets            2702
Clothing_Top base layers    2700
Accessories_Gloves          2699
Accessories_Ski socks       2657

Asia:
Product_Category
Accessories_Ski socks       2685
Accessories_Helmets         2628
Clothing_Suits              2621
Accessories_Gloves          2620
Clothing_Top base layers    2614


### Tópicos de Estratégia (Regional)

**O Alerta da Coreia do Sul**: Na análise regional da Ásia, a Coreia do Sul apresenta uma tendência de queda preocupante, com a receita a diminuir consecutivamente desde 2022 (de 486k€ para 330k€ em 2025). Este é o principal "gap" de performance identificado que pode justificar a saída desta geografia específica.

**Crescimento na América do Norte**: Em contraste com a Ásia, a América do Norte mostra um crescimento sustentado em ambos os países (EUA e Canadá), consolidando-se como a segunda região mais importante e superando a Ásia em volume total de vendas em 2025.

In [ ]:
# ── Coreia do Sul – Tendência de Queda ───────────────────────────────────────
kor = df[(df['Region']=='Asia') & (df['Country']=='South Korea')].groupby('Year').agg(
    Receita = ('Revenue', 'sum'),
    Lucro   = ('Profit',  'sum')
).round(0)
kor['Crescimento (%)'] = kor['Receita'].pct_change().mul(100).map(
    lambda x: f'{x:+.1f}%' if pd.notna(x) else '—')
kor['Receita'] = kor['Receita'].map(lambda x: f'€{x/1e3:.0f}k')
kor['Lucro']   = kor['Lucro'].map(lambda x:   f'€{x/1e3:.0f}k')
print("── Coreia do Sul ─────────────────────────────────────────────")
display(kor)

# ── América do Norte – Crescimento Sustentado ────────────────────────────────
print("\n── América do Norte por País e Ano ───────────────────────────")
na_country = df[df['Region']=='North America'].pivot_table(
    values='Revenue', index='Country', columns='Year', aggfunc='sum'
).round(0)
na_country.columns = na_country.columns.astype(str)
display(na_country.map(lambda x: f'€{x/1e3:.0f}k'))

# ── Comparação Ásia vs América do Norte (Receita Anual) ──────────────────────
print("\n── Ásia vs América do Norte - Receita Anual ─────────────────")
comp = df[df['Region'].isin(['Asia','North America'])].pivot_table(
    values='Revenue', index='Year', columns='Region', aggfunc='sum'
).round(0)
comp.index = comp.index.astype(str)
display(comp.map(lambda x: f'€{x/1e3:.0f}k'))


── Coreia do Sul ─────────────────────────────────────────────


,Receita,Lucro,Crescimento (%)
Year,,,
2020,€321k,€143k,—
2021,€312k,€138k,-2.9%
2022,€486k,€215k,+55.9%
2023,€457k,€204k,-6.0%
2024,€389k,€172k,-14.9%
2025,€331k,€143k,-15.0%



── América do Norte por País e Ano ───────────────────────────


Year,2020,2021,2022,2023,2024,2025
Country,,,,,,
Canada,€299k,€388k,€560k,€594k,€659k,€730k
USA,€393k,€496k,€729k,€799k,€857k,€939k



── Ásia vs América do Norte – Receita Anual ─────────────────


Region,Asia,North America
Year,,
2020,€689k,€693k
2021,€765k,€885k
2022,€1401k,€1290k
2023,€1468k,€1395k
2024,€1443k,€1517k
2025,€1536k,€1669k


---